<!-- OPS:MAT-02:inicio -->
## Esto es “completo” hoy (55 min)

Marca con lo que realmente ocurrió. No hace falta terminar todo.

- [ ] El entorno funciona o abrí `la ruta de rescate docente (no incluida)` y lo declaré.
- [ ] Producto mínimo: Un `model_card.json` que valida contra el schema, con los campos mínimos.
- [ ] Si algo falló: una frase del error, no el traceback completo.
- [ ] Bitácora escrita, aunque diga `lab incompleto`.

Completo ≠ entendido del todo. Incompleto ≠ no puedes seguir. Si la instalación falla después de 10 minutos, usa `la ruta de rescate docente (no incluida)`, conserva el artefacto y marca incompleto.
**Extensión:** Si queda tiempo: prosa completa, tracking externo y documentación adicional.
<!-- OPS:MAT-02:fin -->

# L17 · Model cards causales

**S17 · Lectura: Cap. 19 + Ap. G · 40–55 min**

## La pregunta

¿Puede otra persona saber qué estima el modelo y cuándo no usarlo?

## Qué harás

Documentar estimando, población, supuestos y límites.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Una model card causal validada contra un schema.

> **Idea clave:** Documentar un supuesto no lo vuelve cierto, pero ocultarlo impide revisarlo.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd, json, re, shutil, tempfile, joblib
from pathlib import Path
from jsonschema import Draft202012Validator
rng = setup()
art = a2_fit()
print("A2:", art["estimando"], "·", fmt_ci(art["theta"], art["se"]), "· hash:", art["hash_datos_entrenamiento"][:25], "…")

In [ ]:
# Qué debe quedar visible en una model card
import matplotlib.pyplot as plt
labels = ['estimando', 'población', 'supuestos', 'límites']
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.plot(range(len(labels)), [0] * len(labels), "o-", lw=3)
for i, label in enumerate(labels):
    ax.text(i, 0.08, label, ha="center", va="bottom", fontsize=10)
ax.set_xlim(-.35, len(labels) - .65); ax.set_ylim(-.2, .3); ax.axis("off")
ax.set_title("Qué debe quedar visible en una model card")
plt.tight_layout(); plt.show()

## 1. Tarjeta predictiva frente a tarjeta causal (§19.1–19.3)
Una ficha estándar (Mitchell et al., 2019) describe datos, métricas y usos. Para un modelo **causal** faltan campos sin los cuales el número no se puede interpretar:

In [ ]:
show(pd.DataFrame({
    "campo": ["estimando", "diseño de identificación", "supuestos declarados (con su tipo)", "población de entrenamiento"],
    "¿en la ficha estándar?": ["no", "no", "no", "parcial (como «datos»)"],
    "por qué importa en un modelo causal": [
        "«ATE», «CATE» o «LATE» son números distintos con el mismo nombre de modelo",
        "el mismo código sobre un experimento o sobre datos observacionales significa cosas muy distintas",
        "el efecto es válido solo si valen supuestos que ningún dato confirma: hay que decirlos",
        "el efecto no se traslada automáticamente a otra población (L24)"]}).set_index("campo"))

## 2. El schema de la tarjeta
La tarjeta tiene versión legible por personas (Markdown) y por máquinas (JSON). El **schema** fija la forma; una **puerta** (más estricta que el schema) fija el contenido mínimo del curso.

In [ ]:
TIPOS_S = list(TIPOS); TRATOS_S = list(TRATOS)
CARD_SCHEMA = {
  "type": "object", "additionalProperties": False,
  "required": ["modelo", "version", "estimando", "diseno_identificacion", "supuestos", "poblacion_entrenamiento", "metricas", "no_usar_para", "hash_datos_entrenamiento"],
  "properties": {
    "modelo": {"type": "string", "minLength": 3}, "version": {"type": "string", "minLength": 1},
    "estimando": {"type": "string", "minLength": 3},
    "diseno_identificacion": {"enum": ["experimental", "cuasi-experimental", "observacional"]},
    "supuestos": {"type": "array", "minItems": 1, "items": {
        "type": "object", "required": ["nombre"],
        "properties": {"nombre": {"type": "string", "minLength": 3}, "tipo": {"enum": TIPOS_S}, "tratamiento": {"enum": TRATOS_S},
                       "resultado": {"type": "string"}, "verificado": {"type": "boolean"}, "evidencia": {"type": "string"}},
        "if": {"properties": {"verificado": {"const": True}}, "required": ["verificado"]}, "then": {"required": ["evidencia"]}}},
    "poblacion_entrenamiento": {"type": "object", "required": ["descripcion", "n"], "properties": {"descripcion": {"type": "string", "minLength": 5}, "n": {"type": "integer", "minimum": 1}}},
    "metricas": {"type": "object", "required": ["theta", "ic95"]},
    "no_usar_para": {"type": "array", "minItems": 1, "items": {"type": "string", "minLength": 5}},
    "hash_datos_entrenamiento": {"type": "string", "pattern": "^sha256:[0-9a-f]{64}$"}}}
VAL = Draft202012Validator(CARD_SCHEMA)
def errores(card): return [f"{'/'.join(map(str, e.path)) or '(raíz)'}: {e.message}" for e in VAL.iter_errors(card)]

def puerta_tarjeta(card):
    """Puerta del curso: pasa el schema Y cada supuesto está tipado Y hay al menos un supuesto no observable."""
    err = errores(card)
    for s in card.get("supuestos", []):
        for k in ("tipo", "tratamiento", "resultado"):
            if not s.get(k): err.append(f"supuesto «{s.get('nombre', '?')}» sin «{k}»")
    if not any(s.get("tipo") == "no observable" for s in card.get("supuestos", [])):
        err.append("ningún supuesto es no observable: ¿de verdad no hay ninguno?")
    return err
print("el schema exige solo el nombre de cada supuesto; la puerta exige además tipo, cómo se trató y resultado.")

## 3. Completa tú: la tarjeta de A2 (§19.4)
`construir_tarjeta(art)` debe devolver un diccionario que **pase la puerta**:
- `supuestos`: al menos tres, cada uno con `nombre`, `tipo`, `tratamiento` (cómo se trató) y `resultado`; **al menos uno no observable** (la no confusión condicional lo es);
- `no_usar_para`: al menos tres frases;
- `hash_datos_entrenamiento`: tomado del artefacto (`sha256:` + 64 hexadecimales);
- `metricas`: `theta` e `ic95` del artefacto.

In [ ]:
# ✏️ Completa tú — construir_tarjeta
# Pista: Mira el schema de arriba: tipo ∈ TIPOS_S y tratamiento ∈ TRATOS_S. Un supuesto no observable se «declara y acota».
def construir_tarjeta(art):
    """Devuelve el dict de la tarjeta (ver CARD_SCHEMA) con supuestos tipados y 'no usar para'."""
    tarjeta = {
        "modelo": "dml_plr_ihdp_medium", "version": art.get("version", "1.0"),
        "estimando": art["estimando"],
        "diseno_identificacion": "observacional",
        "supuestos": [],        # TODO: al menos 3, cada uno con nombre, tipo, tratamiento y resultado; al menos uno no observable
        "poblacion_entrenamiento": {"descripcion": "TODO", "n": art["n_train"]},
        "metricas": {"theta": art["theta"], "ic95": art["ic95"]},
        "no_usar_para": [],     # TODO: al menos 3 frases
        "hash_datos_entrenamiento": art["hash_datos_entrenamiento"]}
    if not tarjeta["supuestos"] or not tarjeta["no_usar_para"]:
        raise NotImplementedError("completa supuestos (≥3) y no_usar_para (≥3)")
    return tarjeta

In [ ]:
card = construir_tarjeta(art)
fallos = puerta_tarjeta(card)
print("✔ la tarjeta pasa la puerta" if not fallos else "✘ " + "\n  ".join(fallos))
assert not fallos and len(card["supuestos"]) >= 3 and len(card["no_usar_para"]) >= 3, "completa supuestos (≥3) y no_usar_para (≥3)"

### La puerta falla ante una tarjeta sin supuestos no observables
Probamos la puerta con tres tarjetas incorrectas: sin supuestos no observables, con un supuesto sin tipo, y con `verificado: true` sin evidencia.

In [ ]:
import copy
malas = {}
c1 = copy.deepcopy(card); c1["supuestos"] = [s for s in c1["supuestos"] if s["tipo"] != "no observable"]; malas["sin supuestos no observables"] = c1
c2 = copy.deepcopy(card); c2["supuestos"][0].pop("tipo"); malas["supuesto sin tipo"] = c2
c3 = copy.deepcopy(card); c3["supuestos"][1]["verificado"] = True; malas["verificado sin evidencia"] = c3
c4 = copy.deepcopy(card); c4["extra"] = 1; malas["campo no previsto (additionalProperties)"] = c4
for k, c in malas.items():
    e = puerta_tarjeta(c); print(("✔ rechazada" if e else "✘ ¡pasó!"), "—", k, "→", e[0] if e else "")

**Mira el resultado.** ¿Por qué el schema solo exige `nombre` en cada supuesto, y por qué la puerta del curso pide más?

**Respuesta:**

El schema se mantiene mínimo para que cualquier tarjeta sea *parseable*; la **puerta** codifica la política del curso (cada supuesto tipado, uno no observable al menos). Separarlos permite endurecer la política sin romper tarjetas antiguas y deja claro qué es forma y qué es contenido. Una tarjeta sin ningún supuesto no observable es sospechosa: casi ningún efecto causal observacional lo es.



## 4. Empaquetar el bundle local (§19.4, Listado 19.2)
El bundle contiene **cuatro** piezas: el estimador serializado, el `CausalSchema` (qué covariables espera, en qué orden), la tarjeta en Markdown (`MODEL_CARD.md`) y la tarjeta en JSON validada. Trabajamos en una carpeta temporal.

In [ ]:
def a_markdown(c):
    L = [f"# {c['modelo']} (v{c['version']})", "", "## Estimando", c["estimando"], "", "## Diseño de identificación", c["diseno_identificacion"], "", "## Supuestos"]
    L += [f"- **{s['nombre']}** — {s.get('tipo', '?')} · {s.get('tratamiento', '?')} · {s.get('resultado', '')}" for s in c["supuestos"]]
    L += ["", "## Población de entrenamiento", c["poblacion_entrenamiento"]["descripcion"], f"n = {c['poblacion_entrenamiento']['n']}", "",
          "## Métricas", f"ATE estimado: {c['metricas']['theta']:.3f} (IC 95%: [{c['metricas']['ic95'][0]:.3f}, {c['metricas']['ic95'][1]:.3f}])", "",
          "## No usar este modelo para"] + [f"- {x}" for x in c["no_usar_para"]] + ["", f"Hash de los datos de entrenamiento: `{c['hash_datos_entrenamiento']}`"]
    return "\n".join(L)

def registrar(card, art, destino):
    destino = Path(destino); destino.mkdir(parents=True, exist_ok=True)
    joblib.dump({"theta": art["theta"], "se": art["se"], "ic95": art["ic95"], "hash": art["hash_datos_entrenamiento"], "version": card["version"]}, destino / "dml_estimator.joblib")
    joblib.dump({"covariables": art["covariables"], "soporte": art["soporte"]}, destino / "causal_schema.joblib")
    (destino / "MODEL_CARD.md").write_text(a_markdown(card), encoding="utf-8")
    (destino / "model_card.json").write_text(json.dumps(card, ensure_ascii=False, indent=1), encoding="utf-8")
    return sorted(p.name for p in destino.iterdir())
BUNDLE = Path(tempfile.mkdtemp(prefix="bundle_"))
print("archivos del bundle:", registrar(card, art, BUNDLE))
print(a_markdown(card)[:900], "…")

## 5. Checklist de cinco señales (§19.6)
Convertimos la lista de verificación del capítulo en una función. Cada señal devuelve `True` si **se enciende** (hay un problema).

In [ ]:
def checklist(destino, hash_vigente):
    d = Path(destino); s = {}
    md_ = (d / "MODEL_CARD.md").read_text(encoding="utf-8") if (d / "MODEL_CARD.md").exists() else ""
    js = json.loads((d / "model_card.json").read_text(encoding="utf-8")) if (d / "model_card.json").exists() else None
    s["1 métricas sin diseño ni supuestos"] = ("## Métricas" in md_) and not (("## Diseño de identificación" in md_) and ("## Supuestos" in md_))
    s["2 falta «no usar para»"] = "No usar este modelo para" not in md_
    s["3 falta el CausalSchema en el bundle"] = not (d / "causal_schema.joblib").exists()
    s["4 tarjeta desactualizada tras reentrenar"] = (js is None) or js.get("hash_datos_entrenamiento") != hash_vigente
    s["5 solo Markdown, sin JSON validado"] = (js is None) or bool(errores(js))
    return s
base = checklist(BUNDLE, art["hash_datos_entrenamiento"])
print({k: ("ENCENDIDA" if v else "ok") for k, v in base.items()})

In [ ]:
# Sabotajes: ¿qué señal detecta cada pérdida?
def copia(): d = Path(tempfile.mkdtemp(prefix="sab_")); [shutil.copy2(p, d / p.name) for p in BUNDLE.iterdir()]; return d
res = {}
d = copia(); t = (d / "MODEL_CARD.md").read_text(encoding="utf-8"); (d / "MODEL_CARD.md").write_text(t.split("## No usar")[0], encoding="utf-8"); res["borrar «no usar para»"] = checklist(d, art["hash_datos_entrenamiento"])
d = copia(); (d / "causal_schema.joblib").unlink(); res["borrar el CausalSchema"] = checklist(d, art["hash_datos_entrenamiento"])
d = copia(); (d / "model_card.json").unlink(); res["borrar el JSON"] = checklist(d, art["hash_datos_entrenamiento"])
res["reentrenar con datos nuevos sin actualizar la tarjeta"] = checklist(BUNDLE, "sha256:" + "a" * 64)
show(pd.DataFrame(res).T.replace({True: "ENCENDIDA", False: "·"}))

**Mira el resultado.** ¿Qué sabotaje enciende más de una señal y por qué?

**Respuesta:**

Borrar el JSON enciende la 4 y la 5 (sin JSON no se puede comparar el hash ni validar el schema); las demás encienden una cada una. El chequeo de la señal 4 es el que atrapa el error más común en la práctica: reentrenar con datos nuevos y olvidar regenerar la tarjeta, de modo que dice una cosa y el modelo otra.



## 6. Supuesto y bitácora

In [ ]:
print("bundle completo y tarjeta que pasa la puerta:", not puerta_tarjeta(card) and not any(base.values()))
fila_b = bitacora(
    pregunta="¿Está A2 documentado de modo que otra persona sepa qué estima, bajo qué supuestos y para quién?",
    metodo="Tarjeta causal (Markdown + JSON con schema) y bundle local; puerta de calidad y checklist de cinco señales",
    estimando="ATE de A2 (documentación del artefacto)",
    numero_ic=fmt_ci(art["theta"], art["se"]),
    supuesto=supuesto("observable", "medido", f"la puerta rechaza tarjetas sin supuestos no observables ({len(card['supuestos'])} supuestos declarados) y el checklist no enciende ninguna señal"),
    decision="continuar" if not any(base.values()) else "alertar", lab="L17")

---
## Qué queda para el libro
Quedan en el libro: el Apéndice G completo (plantilla oficial), las datasheets y FactSheets (ejercicios conceptuales) y la validación local de tarjetas. El ejemplo del libro (Card, n = 3003) es ilustrativo; aquí la tarjeta es la de IHDP-medium.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*